In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.metrics import classification_report, accuracy_score


In [ ]:
# Image size and training parameters
img_size = 128
batch_size = 32
epochs = 100
dataset_path='autism_dataset'


In [10]:
# Data generators for train and validation
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,  # 80% train, 20% validation
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

train_data = datagen.flow_from_directory(
    dataset_path,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='binary',
    subset='training'
)

val_data = datagen.flow_from_directory(
    dataset_path,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='binary',
    subset='validation',
)


Found 2669 images belonging to 2 classes.
Found 667 images belonging to 2 classes.


In [11]:
# Simple CNN architecture
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(img_size, img_size, 3)),
    MaxPooling2D(2, 2),
    
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])


C:\Users\dsair\AppData\Roaming\Python\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [12]:
# Compile model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(train_data, epochs=epochs, validation_data=val_data)

C:\Users\dsair\AppData\Roaming\Python\Python311\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 74s 864ms/step - accuracy: 0.8577 - loss: 0.9104 - val_accuracy: 0.8801 - val_loss: 0.3772
Epoch 2/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 25s 300ms/step - accuracy: 0.8740 - loss: 0.4023 - val_accuracy: 0.8801 - val_loss: 0.3988
Epoch 3/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 26s 312ms/step - accuracy: 0.8851 - loss: 0.3835 - val_accuracy: 0.8801 - val_loss: 0.3721
Epoch 4/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 27s 316ms/step - accuracy: 0.8822 - loss: 0.3908 - val_accuracy: 0.8801 - val_loss: 0.3819
Epoch 5/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 26s 313ms/step - accuracy: 0.8808 - loss: 0.3854 - val_accuracy: 0.8801 - val_loss: 0.3692
Epoch 6/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 26s 314ms/step - accuracy: 0.8725 - loss: 0.3951 - val_accuracy: 0.8801 - val_loss: 0.3704
Epoch 7/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 33s 391ms/step - accuracy: 0.8745 - loss: 0.3843 - val_accuracy: 0.8801 - val_loss: 0.3721
Epoch 8/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 39s 459ms/step - accuracy: 0.8698 - loss: 0.4015 - 

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)
cm = confusion_matrix(y_true, y_pred_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, cmap='plasma', fmt='d', xticklabels=lb.classes_, yticklabels=lb.classes_)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# Evaluating Accuracy

In [13]:
# Evaluate accuracy
val_loss, val_acc = model.evaluate(val_data)
print(f"\n Final Validation Accuracy: {val_acc * 100:.2f}%")


21/21 ━━━━━━━━━━━━━━━━━━━━ 2s 76ms/step - accuracy: 0.8847 - loss: 0.5224

 Final Validation Accuracy: 87.56%


In [14]:
# Predict and evaluate performance
y_true = val_data.classes
y_pred_prob = model.predict(val_data)
y_pred = (y_pred_prob > 0.5).astype('int32').flatten()

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))


21/21 ━━━━━━━━━━━━━━━━━━━━ 2s 76ms/step

Classification Report:

              precision    recall  f1-score   support

           0       0.88      0.98      0.93       587
           1       0.09      0.01      0.02        80

    accuracy                           0.87       667
   macro avg       0.49      0.50      0.48       667
weighted avg       0.78      0.87      0.82       667



In [8]:
model.save('model.keras')
